In [6]:
from pathlib import Path 
import json, inspect, time, warnings 
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 
import torch 
from torch import nn 
from scipy.stats import qmc, norm 

try: 
    import QuantLib as ql 
except ImportError as exc: 
    raise ImportError("Install QuantLib") from exc 

DEVICE = torch.device('cpu')
torch.set_num_threads(1)
print('torch:', torch.__version__, '| QuantLib:', getattr(ql, '__version__', 'unknown'))

torch: 1.12.1 | QuantLib: 1.43


In [7]:
FEATURES = ['S0', 'K', 'T', 'r', 'v0', 'kappa', 'theta', 'sigma', 'rho']
BOUNDS = {
    'S0':(0.6, 1.4),
    'T': (0.1, 1.4),
    'r': (0.0, 0.1),
    'v0': (0.05, 0.5),
    'kappa': (0.0, 2.0),
    'theta': (0.0, 0.5),
    'sigma': (0.0, 0.5),
    'rho': (-0.95, 0.0)
}
TRAIN_N = 12000
MODEL_SEEDS = (42, 43, 44)
K_FIXED = 1.0 
Q_FIXED = 0.0 
INTEGRATION_ORDER = 128 
VALUATION = ql.Date(2, ql.January, 2026)
SEED_BASE = 20261119
REFERENCE_CHECK_ORDER = 64 
REFERENCE_CHECK_TOL = 1e-5
N_BOOTSTRAP = 1000 
N_PER_REGIME = 200 
N_CONTROL = 500 
INCLUDE_OPTIONAL_MLP = True 

ROOT = Path.cwd()
MODEL_COMPARISON_MODEL_DIR = ROOT / 'experiment_training_size_results' / f'n{TRAIN_N}' / 'models' 
ARCHITECTURE_MODEL_DIR = ROOT / 'experiment_training_size_results' / f'n{TRAIN_N}' / 'models'
EXTRA_MODEL_DIRS = [ROOT, Path('/mnt/data')]
OUTPUT_DIR = ROOT / 'experiment_training_size_results_v2' / f'n{TRAIN_N}'/ f'nregime{N_PER_REGIME}_control{N_CONTROL}'
DATA_DIR = OUTPUT_DIR / 'regime_data'
FIGURE_DIR = OUTPUT_DIR / 'figures'
for folder in (DATA_DIR, FIGURE_DIR): 
    folder.mkdir(parents=True, exist_ok=True)
print('Output folder: ', OUTPUT_DIR.resolve())
print('Primary checkpoint folder: ', MODEL_COMPARISON_MODEL_DIR)

Output folder:  /Users/john.madhu_/Desktop/MSC_thesis/src/heston/experiment_training_size_results_v2/n12000/nregime200_control500
Primary checkpoint folder:  /Users/john.madhu_/Desktop/MSC_thesis/src/heston/experiment_training_size_results/n12000/models


In [8]:
REGIMES = [
    ('Control / full domain', {}), 
    ('Matched base / full domain', {}), 
    ('Strong negative correlation', {'rho': (-0.95, -0.70)}),
    ('Weak negative correlation', {'rho': (-0.20, 0.00)}), 
    ('High vol-of-vol', {'sigma':(0.35, 0.50)}), 
    ('Low vol-of-vol', {'sigma': (0.00, 0.15)}), 
    ('Short maturity', {'T': (0.10, 0.35)}), 
    ('Long maturity', {'T': {1.05, 1.40}}), 
    ('Out-of-the-money-calls', {'S0': (0.60, 0.85)}), 
    ('At-the-money-calls', {'S0': (0.95, 1.05)}), 
    ('In-the-money-calls', {'S0': (1.15, 1.40)}), 
    ('High initial variance', {'v0': (0.35, 0.50)}), 
    ('Low initial variance', {'v0': (0.05, 0.15)}), 
    ('Fast mean reversion', {'kappa': (1.50, 2.00)}), 
    ('Slow mean reversion', {'kappa': (0.00, 0.50)}), 
    ('High long-run variance', {'theta': (0.35, 0.50)}), 
    ('Low long-run variance', {'theta': (0.00, 0.15)}),
    ('Strong leverage + high vol-of-vol', {'rho': (-0.95, -0.70), 'sigma': (0.35, 0.50)}),
    ('Short maturity + OTM ', {'T': (0.10, 0.35), 'S0': (0.60, 0.85)}), 
    ('High variance + strong leverage', {'v0': (0.35, 0.50), 'rho': (-0.95, -0.70)}),
]

VARIABLE_FEATURES = [x for x in FEATURES if x != 'K']
assert len(VARIABLE_FEATURES) == 8 
for name, overrides in REGIMES: 
    for parameter, (lo, hi) in overrides.items(): 
        assert parameter in BOUNDS, (name, parameter)
        full_lo, full_hi = BOUNDS[parameter]
        assert full_lo <= lo < hi <= full_hi, (name, parameter)
print(f'{len(REGIMES)} regimes; {N_CONTROL + (len(REGIMES) - 1) * N_PER_REGIME: ,} new reference labels requested (including separate matched base). ')

20 regimes;  4,300 new reference labels requested (including separate matched base). 


In [9]:
class MLP(nn.Module): 
    def __init__(self, hidden_units):
        super().__init__()
        layers = []
        width = len(FEATURES)
        for hidden_width in hidden_units: 
            layers.extend([
                nn.Linear(width, hidden_width), 
                nn.ReLU()])
            width = hidden_width
        layers.append(nn.Linear(width, 1))
        self.net = nn.Sequential(*layers)
    def forward(self,x):
        return self.net(x)
    
def find_checkpoint(stem, default_dir): 
    dirs = [default_dir, MODEL_COMPARISON_MODEL_DIR, ARCHITECTURE_MODEL_DIR] + EXTRA_MODEL_DIRS
    filenames = [f'{stem}.pt', f'{stem}(1).pt', f'{stem} (1).pt']
    for folder in dirs: 
        for filename in filenames: 
            candiate = folder / filename 
            if candiate.is_file(): 
                return candiate
    return None 


def load_frozen_checkpoints(path, seed, model_name, expected_architecture=None):
    kwargs = {'map_location': 'cpu'}
    if 'weights_only' in inspect.signature(torch.load).parameters: 
        kwargs['weights_only'] = False 
    checkpoint = torch.load(path, **kwargs)
    if list(checkpoint['features']) != FEATURES or int(checkpoint['train_n']) != TRAIN_N: 
        raise ValueError(f'Feature/training-size mismatch in {path}')
    if int(checkpoint['model_seed']) != seed: 
        raise ValueError(f'Seed mismatch in {path}')
    units = tuple(int(x) for x in checkpoint['hidden_units'])
    if expected_architecture is not None and units != tuple(expected_architecture):
        raise ValueError(f'Architecture mismatch in {path}: {units}')
    if model_name == 'Hybrid BS-MLP': 
        if checkpoint.get('target') != 'residual' or checkpoint.get('bs_volatility') != 'expected_average_variance': 
            raise ValueError(f'Not a compatible BS-residual checkpoint: {path}')
    else: 
        if checkpoint.get('target', 'price') != 'price': 
            raise ValueError(f'Not a price-trained MLP: {path}')
    model = MLP(units).to(DEVICE)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    mu_x = np.asarray(checkpoint['x_mean'], dtype=np.float(64).reshape(-1))
    sd_x = np.asarray(checkpoint['x_scale'], dtype=np.float(64).reshape(-1))
    mu_y = np.asarray(checkpoint['y_mean'], dtype=np.float(64).reshape(-1)[0])
    sd_y = np.asarray(checkpoint['y_scale'], dtype=np.float(64).reshape(-1)[0])
    if (mu_x.size != len(FEATURES) or sd_x.size != len(FEATURES)
        or not np.isfinite(mu_x).all() or not np.isfinite(sd_x).all()
        or (sd_x <= 0).any() or sd_y <= 0):
        raise ValueError(f'Invalid saved scalers in {path}')
    return {'model': model, 'x_mean':mu_x, 'x_scale': sd_x, 
            'y_mean': mu_y, 'y_scale': sd_y, 'path': str(path), 'units': units}

MODELS = {}
model_specs = [('Hybrid BS-MLP', 'model_Hybrid', MODEL_COMPARISON_MODEL_DIR, (128, 128, 128), True)] 
if INCLUDE_OPTIONAL_MLP: 
    model_specs += [
        ('MLP Baseline', 'model_Baseline', ARCHITECTURE_MODEL_DIR, (128, 128, 128), False),
        ('MLP Wide', 'model_Wide', ARCHITECTURE_MODEL_DIR, (128, 128, 128), False)
    ]
for name, stem, folder, units, required in model_specs: 
    paths = [find_checkpoint(f'{stem}_seed{seed}', folder) for seed in MODEL_SEEDS]
    if any(p is None for p in paths):
        if required:
            raise FileNotFoundError(f'Missing Hybrid checkpoints: {paths}. '
                 'Copy model_Hybrid_seed42/43/44.pt into Experiment 4 model folder, '
                 'or set EXTRA_MODEL_DIRS to the folder')
        print('Skipping optional model (some checkpoints missing): ', name)
        continue
    MODELS[name] = {
        seed:load_frozen_checkpoints(path, seed, name, units)
        for seed, path in zip(MODEL_SEEDS, paths)
    } 
print('Evaluating:', ['Black-Scholes'] + list(MODELS))
print('All models are frozen; no training occurs in this notebook,')

FileNotFoundError: Missing Hybrid checkpoints: [None, None, None]. Copy model_Hybrid_seed42/43/44.pt into Experiment 4 model folder, or set EXTRA_MODEL_DIRS to the folder

In [ ]:
def slug(label): 
    import re 
    return re.sub(r'[^a-z0-9]+','_', label.lower()).strip('_')

def sample_regime(name, bounds_override, n, seed, shared=None): 
    ranges = dict(BOUNDS)
    ranges.update(bounds_override)
    lower = np.array([ranges[k][0] for k in VARIABLE_FEATURES], dtype=float)
    upper = np.array([ranges[k][1] for k in VARIABLE_FEATURES], dtype=float)
    if shared is None: 
        unit = qmc.LatinHypercube(d=len(VARIABLE_FEATURES), seed=seed).random(n)
    else: 
        unit = np.asarray(shared, dtype=float)
        assert unit.shape == {n, len(VARIABLE_FEATURES)}
    frame = pd.DataFrame(qmc.scale(unit, lower, upper), columns=VARIABLE_FEATURES)
    frame['K'] = K_FIXED
    frame['T'] = np.maximum(1, np.round(frame['T'] * 365).astype(int)) / 365.0
    frame = frame[FEATURES].copy()
    frame.insert(0, 'case_id', [f'{slug(name)}_{i:04d}' for i in range(n)])
    frame.insert(1, 'regime', name)
    frame.insert(2, 'design_id', np.arange(n, dtype=int))
    frame['feller_satisfied'] = 2*frame['kappa']*frame['theta'] >= frame['sigma']**2
    return frame 

def heston_reference_price(row): 
    ql.Settings.instance().evaluationDate = VALUATION
    dc = ql.Actual365Fixed()
    maturity = VALUATION + max(1, int(round(float(row['T']) * 365)))
    rf = ql.YieldTermStructureHandle(ql.FlatForward(VALUATION, float(row['r']), dc))
    div = ql.YieldTermStructureHandle(ql.FlatForward(VALUATION, Q_FIXED, dc))
    spot = ql.QuoteHandle(ql.SimpleQuote(float(row['S0'])))
    process = ql.HestonProcess(rf, div, spot, float(row['v0']), float(row['kappa']), 
        float(row['theta']), float(row['sigma']), float(row['rho']))
    model = ql.HestonModel(process)
    option = ql.VanillaOption(ql.PlainVanillaPayoff(ql.Option.Call, float(row['K'])), 
                              ql.EuropeanExercise(maturity))
    option.setPricingEngine(ql.AnalyticHestonEngine(model, INTEGRATION_ORDER))
    value = float(option.NPV())
    lo = max(float(row['S0']) - float(row['K']) * np.exp(-float(row['r']) * float(row['T'])), 0)
    if not np.isfinite(value) or value < lo-1e-4 or value > float(row['S0']) + 1e-4: 
        raise ValueError(f'Invalid/non-finite Heston reference: {value}')
    return float(option.NPV())

GENERATION_CONFIG = {
    'experiment': 'robustness_v2_matched', 
    'train_n': TRAIN_N, 
    'bounds': BOUNDS, 
    'regimes': REGIMES,
    'n_per_regime': N_PER_REGIME, 
    'n_control': N_CONTROL, 
    'seed_base': SEED_BASE, 
    'features': FEATURES, 
    'K_FIXED': K_FIXED, 
    'q_fixed': Q_FIXED, 
    'integration_order': INTEGRATION_ORDER, 
    'reference_check_order': REFERENCE_CHECK_ORDER, 
    'reference_check_tolerance': REFERENCE_CHECK_TOL,
    'valuation_date:': '2026-01-02', 
    'engine': 'QuantLib.AnalayticHestonEngine'
    'design': 'independent_control_plus_common_lhs_matched_regimes', 
}

config_file = DATA_DIR/"generation_config.json"
if config_file.exists(): 
    previous = json.loads(config_file.read_text())
    if previous["config"] != GENERATION_CONFIG: 
        raise RuntimeError(
            "Experiment settings have changed"
        )
else: 
    config_file.write_text(
        json.dumps(
            {"config": GENERATION_CONFIG}, 
            indent = 2
        )
    )

all_frames = []
for i, (name, override) in enumerate(REGIMES): 
    n = N_CONTROL if i == 0 else N_PER_REGIME
    path = DATA_DIR/f'{i:02d}_{slug(name)}.csv'
    if path.exists(): 
        df = pd.read_csv(path)
        required = FEATURES + ['price', 'reference_status', 'case_id', 'regime',
                               'design_id', 'reference_order64', 'reference_abs_diff_64_128', 
                               'reference_check_pass']
        if len(df) != n or set(required) - set(df.columns): 
            raise ValueError(f'Incompatible cache: {path}')
        if not df['regime'].eq(name).all(): 
            raise ValueError(f'Wrong regime in cache: {path}')
    else: 
        df = sample_regime(name, override, n, SEED_BASE+i * 101, 
                           shared=None if i == 0 else shared_unit)
        price128. price64, status = [], [], []
        t0 = time.perf_counter()
        for row in df.to_dict()
        for row in df.to_dict('records'): 
            try: 
                p128=heston_reference_price(row, INTEGRATION_ORDER)
                price128.append(p128)
                status.append('ok')
            except Exception as exc: 
                price128.append(np.nan)
                price64.append(np.nan)
                status.append(f'{type(exc).__name__}: {str(exc)[:170]}')
                continue 
            try: 
                price64.append(heston_reference_price(row, REFERENCE_CHECK_ORDER))
            except Exception as exc: 
                price64.append(np.nan)
                status[-1]='ok; order64 failure: ' +type(exc).__name__
        df['price']=price128
        df['reference_order64']=price64
        df['reference_abs_diff_64_128'] = np.abs(df['price']-df['reference_order64'])
        df['reference_check_pass'] = df['reference_abs_diff_64_128'] <= REFERENCE_CHECK_TOL 
        df['reference_status'] = ['ok' if s=='ok' or s.startswith('ok;') else s for s in status]
        df.to_csv(path, index=False)
        n_ok = int(df['reference_status'].eq('ok').sum())
        n_check = int(df['reference_check_pass'].fillna(False).sum())
        print(f'{i+1:02d}/{len(REGIMES)}{name}: valid {n_ok}/{n}, convergence {n_check}/{n}'
              f'{time.perf_counter()-t0:.1f}s', flush=True)
    all_frames.append(df)
cases = pd.concat(all_frames, ignore_index=True)
assert cases['case_id'].is_unique 
failures = cases.loc[cases['reference_status']!='ok'].copy()
failures.to_csv(OUTPUT_DIR/'reference_failures.csv', index=False)

coverage = cases.groupby('regime', sort=False).agg(
    requested=('case_id', 'size')
    valid = ('reference_status', lambda s: int(s.eq('ok').sum())), 
    reference_check_passed = ('reference_check_pass', lambda s:int(s.fillna(False).sum())), 
    max_reference_diff=('reference_abs_diff_64_128', 'max')
    feller_satisfied_pct = ('feller_satisfied', lambda s: float(s.mean()*100)),
).reset_index()

coverage['reference_failure_pct'] = 100 * (1-coverage['valid']/coverage['requested'])
coverage['reference_unstable_pct'] = 100 * (1-coverage['reference_check_passed']/coverage['requested'])
coverage.to_csv(OUTPUT_DIR/'reference_coverage.csv', index=False)
if len(failures):
    warnings.warn(f'{len(failures)} reference labels failed') 
if coverage['reference_check_passed'] < 0.95 * coverage['requested'].any(): 
    raise RuntimeError("More than 5(%) failed")
if (coverage['valid'] < 0.95*coverage['requested']).any():
    raise RuntimeError("More than 5(%) failed")
if not cases ['reference_check_pass'].fillna(False).all()
    warnings.warn('Flagged for sensivitive analysis')
cases=cases.loc[cases['reference_status']=='ok'].reset_index(drop=True)

paird_base=cases.loc[cases['regime']=='Matched base/full domain'].set_index('design_id')
for name, override in REGIMES[2:]: 
    stress = cases.loc[cases['regime']==name].set_index('design_id')
    unaffected=[f for f in VARIABLE_FEATURES if f not in override]
    assert np.array_equal(paird_base.loc[common,unaffected].to_numpy(), 
                          stress.loc[common,unaffected].to_numpy()), name 
print('Validated matching')
    

In [ ]:
def black_scholes_price(frame): 
    S0 = frame['S0'].to_numpy()
    K = frame['K'].to_numpy()
    T = frame['T'].to_numpy()
    r = frame['r'].to_numpy()
    v0 = frame['v0'].to_numpy()
    kappa = frame['kappa'].to_numpy()
    theta = frame['theta'].to_numpy()

    a = kappa * T 
    weight = np.divide(-np.expm1(-a), a, out=np.ones_like(a), where=a != 0)
    variance = theta + (v0 - theta) * weight 
    vol = np.sqrt(variance)

    d1 = (np.log(S0 / K) + (r + 0.5 * vol**2) * T) / (vol * np.sqrt(T))
    d2 = d1 - vol * np.sqrt(T)
    return S0 * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)


def frozen_mlp_predict(prepared, df): 
    X = df[FEATURES].to_numpy(dtype=np.float64)
    x = (X-prepared['x_mean'])/prepared['x_scale']
    net = prepared['model']
    net.eval()
    with torch.inference_mode(): 
        scaled_pred = net(torch.from_numpy(x.astype(np.float32))).cpu().numpy().ravel()
    return scaled_pred * prepared['y_scaled']+prepared['y_mean']


actual = cases['price'].to_numpy(dtype=float)
bs_pred = black_scholes_price(cases)
records = []

def append_predictions(model_name, seed, preds): 
    preds = np.asarray(preds, dtype=float).ravel()
    if preds.shape != actual.shape or not np.isfinite(preds).all():
        raise ValueError(f'Invalid predicitions for {model_name} seed={seed}')
    frame = cases [['case_id', 'regime', 'design_id']+FEATURES+['price','feller_satisfied','reference_abs_diff_64_128', 'reference_check_pass']].copy()
    frame['model'] = model_name
    frame['model_seed'] = seed 
    frame['predicted_price'] = preds 
    frame['signed_error'] = preds - actual 
    frame['relative_error_pct'] = np.where(actual >= 0.01, 100*np.abs(preds-actual)/actual, np.nan)
    frame['absolute_error_as_pct_spot'] = 100 * np.abs(preds-actual)/frame['S0'].to_numpy()
    lower = np.maximum(frame['S0'] - frame['K'] * np.exp(-frame['r']*frame['T']),0.)
    upper = frame['S0']
    frame['below_lower_bound'] = preds < lower - 1e-8
    frame['above_upper_bound'] = preds > upper + 1e-8
    frame['violates_price_bounds'] = frame['below_lower_bound'] | frame['above_upper_bound']
    records.append(frame)

append_predictions('Black-Scholes', -1, bs_pred)
for model_name, seed_mapping in MODELS.items():
    for seed, prepared in seed_mapping.items(): 
        preds = frozen_mlp_predict(prepared, cases)
        if model_name == 'Hybrid BS-MLP': 
            preds = bs_pred + preds 
        append_predictions(model_name, seed, preds)

predictions = pd.concat(records, ignore_index=True)
predictions.to_csv(OUTPUT_DIR/'robustness_predictions.csv', index=False)
print(f'{len(predictions):, } saved contract/mode/seed evaluations')

In [ ]:
def calculate_metrics(g): 
    e = g['signed_error'].to_numpy(dtype=float) 
    ae = np.abs(e) 
    rel = g['relative_error_pct'].to_numpy(dtype=float)
    return pd.Series({
        'n_contracts': len(g), 
        'mae': float(ae.mean()), 
        'rmse': float(np.sqrt(np.mean(e**2))),
        'median_absolute_error': float(np.median(ae)), 
        'p90_absolute_error': float(np.percentile(ae,90)), 
        'max_absolute_error': float(ae.max()), 
        'mean_absolute_error_as_pct_spot': float(g['absolute_error_as_pct_spot'].mean()), 
        'mean_relative_error_pct_price_ge_001': float(np.nanmean(rel)) if np.isfinite(rel).any() else np.nan,
        'n_relative_error_eligible': int(np.isfinite(rel).sum()), 
        'price_bound_violation_pct': 100*float(g['violates_price_bounds'].mean()), 
        'n_price_bound_violations': int(g['violates_price_bounds'].sum()),
    })
trial_metrics = predictions.groupby(['regime', 'model', 'model_seed'], sort=False).appy(
    calculate_metrics, include_groups=False
).reset_index()
trial_metrics.to_csv(OUTPUT_DIR/'robustness_trials.csv', index=False)
numeric_metrics = ['mae', 'rmse', 'meidan_absolute_error', 'p90_absolute_error', 
                   'max_absolute_error', 'mean_absolute_error_as_pct_spot', 
                   'mean_relative_error_pct_price_ge_001', 'price_bound_violation_pct']
aggregate = trial_metrics.groupby(['regime', 'model'], sort=False).agg(
    repeats=('model_seed', 'size'), 
    n_contracts = ('n_contracts', 'first'), 
    n_relative_error_eligible = ('n_relative_error_eligible', 'first'), 
    **{f'{m}_mean': (m, 'mean') for m in numeric_metrics}, 
    **{f'{m}_std': (m, 'std') for m in numeric_metrics},
).reset_index()
aggregate.to_csv(OUTPUT_DIR/'robustness_summary.csv', index=False)

hybrid = aggregate.query('model == "Hybrid BS-MLP"').set_index('regime')
control_mae = float(hybrid.loc['Control / full domain', 'mae_mean'])
comparison = aggregate[['regime', 'model', 'mae_mean', 'p90_absolute_error_mean',
                        'max_absolute_error_mean', 'price_bound_violation_pct_mean']].copy()
comparison = comparison.merge(hybrid[['mae_mean']].rename(columns={'mae_mean':'hybrid_mae'}), left_on='regime', right_index=True)
comparison['hybrid_mae_improvement_pct'] = 100 * (1-comparison['hybrid_mae']/comparison['mae_mean'])
comparison.to_csv(OUTPUT_DIR/'robustness_model_comparison.csv', index=False)

print('\nHybrid regime performance (highest values deserve follow-up):')
print(aggregate[aggregate.model=='Hybrid BS-MLP'][
    ['regime', 'mae_mean', 'mae_std', 'p90_absolute_error_mean',
     'max_absolute_error_mean', 'price_bound_violation_pct_mean']
].sort_values('mae_mean', ascending=False).to_string(index=False, float_format=lambda x:f'{x:.6g}'))
print('\n Note: std reflects model initialization seeds only (three for Hybrid)')

per_case = predictions.groupby(['regime', 'design_id', 'case_id', 'model'], sort=False).agg(
    abs_error=('absolute_error', 'mean'), 
    signed_error = ('signed_error', 'mean'), 
    true_price = ('price', 'first'), 
    reference_check_pass = ('reference_check_pass', 'first')
    feller_satisfied =('feller_satisfied', 'first')
).reset_index() 

rng = np.random.default_rng(SEED_BASE+991)
boot_rows = []
for (regime, model_name), group in per_case.groupby(['regime', 'model'], sort=False): 
    vals = group['abs_error'].to_numpy(dtype=float)
    samples = rng.integers(0, len(vals), (N_BOOTSTRAP, len(vals)))
    estimates = vals[samples].mean(axis=1)
    boot_rows.append({'regime':regime, 'model':model_name, 'n_cases': len(vals), 
        'mae_contact_bootstrap_lower95':np.percentile(estimates,2.5),
        'mae_contact_bootstrap_upper95':np.percentile(estimates,97.5)})
    bootstrap_ci = pd.DataFrame(boot_rows)
    bootstrap_ci.to_csv(OUTPUT_DIR/'robustness_contract_bootstrap_ci.csv', index=False)
    aggregate=aggregate.merge(bootstrap_ci,on=['regime','model'], how='left')
    aggregate.to_csv(OUTPUT_DIR/'robustness_summary.csv',index=False)

paired_model_rows = []
for regime, reg_df in per_case.groupby('regime', sort=False): 
    wide=reg_df.pivot(index='case_id', columns='model', values='abs_error')
    hybrid_errors = wide['Hybrid BS-MLP'].to_numpy() 
    for comparator in [m for m in wide.columns if m!='Hybrid BS-MLP']:
        comp=wide[comparator].to_numpy()
        diff=comp-hybrid_errors
        indexes=rng.integers(0,len(diff),(N_BOOTSTRAP, len(diff)))
        mean_comp=comp[indexes].mean(axis=1)
        reductions=100*(1-hybrid_errors[indexes].mean(axis=1)/mean_comp)
        paired_model_rows.append({
            'regime':regime,
            'comparator':comparator, 
            'n_cases':len(diff), 
            'hybrid_minus_comparator_mae':-float(diff.mean()), 
            'hybrid_improvement_pct_bootstrap_lower95': np.percentile(reductions,2.5), 
            'hybrid_improvement_pct_bootstrap_upper95': np.percentile(reductions,97.5),
            'hybrid_better_fraction_of_contracts': float(np.mean(hybrid_errors < comp)) 
        })
paired_model_comparison = pd.DataFrame(paired_model_rows)
paired_model_comparison.to_csv(OUTPUT_DIR/'robustness_paired_model_comparison.csv', index=False)

paired_stress_rows = []
for model_name, mg in per_case.groupby('model', sort=False): 
    base=(mg.loc[mg['regime']=='Matched base/full domain'].set_index('design_id')['abs_error'])
    for regime in [n for n,_ in REGIMES[2:]]:
        stress = mg.loc[mg['regime']==regime].set_index('design_id')['abs_error']
        common=base.index.intersection(stress.index)
        d = (stress.loc[common]-base.loc[common]).to_numpy(dtype=float)
        ix = rng.integers(0, len(d), (N_BOOTSTRAP, len(d)))
        bs=d[ix].mean(axis=1)
        paired_model_rows.append({
            'model':model_name,
            'regime': regime,
            'n_matched_cases':len(d), 
            'mae_stress_minus_matched_base':float(d.mean()), 
            'delta_lower95': np.percentile(bs,2.5), 
            'delta_upper95': np.percentile(bs, 97.5) 
        })
paired_stress = pd.DataFrame(paired_stress_rows)
paired_stress.to_csv(OUTPUT_DIR/'robustness_paired_stress_vs_base.csv', index=False)


feller_summary = per_case.groupby(['regime', 'model'], sort=False).apply(
    lambda df:pd.Series({'n_reference_unstable': int((~df['reference_check_pass'].astype(bool)).sum())}),
    include_groups=False).reset_index()
feller_summary.to_csv(OUTPUT_DIR/'reference_flags_by_model.csv',index=False)

feller_error_summary = per_case.groupby(['regime', 'model', 'feller_satisfied'], sort=False)
    .agg(n_cases=('case_id', 'size'), 
         mae=('abs_error', 'mean'),
         median_abs_error=('abs_error', 'median'), 
         p90_abs_error=('abs_error', lambda s:float(s.quantile(.90))), 
         max_abs_error=('abs_error', 'max')).reset_index())
feller_error_summary['few_cases_warning']=feller_error_summary['n_cases']<20
feller_error_summary.to_csv(OUTPUT_DIR/'robustness_by_feller_condition.csv',index=False)

print('\n Hybrid vs MLP Baseline: paired improvement (%) across regimes: ')
print(paired_model_comparison.loc[paired_model_comparison['comparator']=='MLP Baseline', 
      ['regime', 'hybrid_improvement_pct', 'hybrid_improvement_pct_bootstrap_lower_95', 
       'hybrid_improvement_pct_bootstrap_upper95']].to_string(index=False, float_format=lambda x:f'x:.3g'))
print('Contract-level bootstrap ci are separate from 3 seed models')


In [ ]:
hybrid_preds = predictions.query('model == "Hybrid BS-MLP"').copy()
worst = hybrid_preds.groupby(['case_id', 'regime'], sort=False).agg(
    mean_absolute_error = ('absolute_error', 'mean'), 
    worst_seed_absolute_error = ('absolute_error', 'mean'), 
    true_price=('price', 'first'), 
    mean_predicted_price = ('predicted_price', 'mean')
    feller_satisfied = ('feller_satisfied', 'first'), 
    **{k:(k, 'first') for k in FEATURES}
).reset_index().sort_values('mean_absolute_error', ascending=False)
worst.head(50).to_csv(OUTPUT_DIR/'hybrid_worst_50_cases.csv', index=False)
print('Largest mean-across-seed Hybrid pricing errors: ')
print(worst.head(12).to_string(index=False, float_format=lambda x:f'{x:.6g}'))

model_order = ['Hybrid BS-MLP', 'MLP Baseline', 'MLP Wide', 'Black-Scholes']
plot_models = [x for x in model_order if x in set(aggregate['model'])]
regime_order = [name for name, _ in REGIMES] 

fig, ax = plt.subplots(figsize=(15,8))
x = np.arange(len(regime_order))
width = 0.8/len(plot_models)
for j, name in enumerate(plot_models): 
    sub = aggregate[aggregate['model']==name].set_index('regime').reindex(regime_order)
    ax.bar(x-0.4+(j+0.5)*width, sub['mae_mean'], width=width, label=name)
ax.set_yscale('log')
ax.set_ylabel('Test MAE (normalized option price units; logarithmic scale)')
ax.set_xlabel('Financial parameter regime')
ax.set_title('Robustness: accuracy across Heston regimes')
ax.set_xticks(x, regime_order, rotation=55, ha='right')
ax.grid(axis='y', alpha=0.25)
ax.legend()
fig.tight_layout()
fig.savefig(FIGURE_DIR/'robustness_mae_by_regime.png', dpi=220)
plt.show()
plt.close(fig)


fig, ax = plt.subplots(figsize=(15,8))
hybrid_sorted = hybrid.reindex(regime_order).sort_values('p90_absolute_error_mean', ascending=False)
ax.barh(hybrid_sorted.index[::-1], hybrid_sorted['p90_absolute_error_mean'][::-1])
ax.set_xlabel('90th percentile of absolute pricing error (normalized option price units)')
ax.set_title('Hybrid BS-MLP: upper-tail errors by regime')
ax.grid(axis='x', alpha=0.25)
fig.tight_layout()
fig.savefig(FIGURE_DIR/'hybrid_p90_by_regime.png', dpi=220)
plt.show()
plt.close(fig)

fig, ax = plt.subplots(figsize=(15,8))
for name in plot_models: 
    a = predictions[(predictions.model==name) & (predictions.regime=='Control/full domain')]
    if a.empty: continue 
    ax.hist(a['absolute_error'].to_numpy(), bins=40, alpha=.45, density=True)
ax.set_yscale('log')
ax.set_xlabel('Absolute error (normalized option price units)')
ax.set_ylabel('Density(log scale)')
ax.set_title('Error distributions on the independent full-domain control set')
ax.legend()
fig.tight_layout()
fig.savefig(FIGURE_DIR/'control_absolute_distribution.png', dpi=220)
plt.show()
plt.close(fig)

report = {
    'experiment': 5, 
    'research_questions': ['RQ1', 'RQ4'], 
    'selected_model': 'Hybrid BS-MLP (selected previously by validation MAE)', 
    'model_seeds': MODEL_SEEDS, 
    'train_n': TRAIN_N, 
    'labels': GENERATION_CONFIG, 
    'models_evaluated': ['Black-Scholes']+list(MODELS)
    'model_checkpoint':{name:{str(seed):item['path'] for seed, item in seeds.items()} for name, seeds in MODELS.items()},
    'test_selection': 'Frozen before robustness evaluation; no tuning on robustness labels', 
    'interpretation_notes': [
        'All default regimes are inside the training bounds but use shifted sampling distribution. ', 
        'Reference failures are reported, not silently dropped. '
        'Seed SD is model initialization variability, not sampling uncertainty', 
        ''
    ] 
}
(OUTPUT_DIR/'robustness_config.json').write_text(json.dumps(report, indent=2, default=str))
print('\nSaved all files to:', OUTPUT_DIR.resolve())

if not paired_stress.empty:
    fig, ax = plt.subplots(figsize=(12,6))
    h=paired_stress.loc[paired_stress['model']=='Hybrid BS-MLP'].copy()
    h=h.set_index('regime').reindex([n for n,_ in REGIMES[2:]])
    vals=h['mae_stress_minus_matched_base'].to_numpy(dtype=float)
    ax.barh(np.arange(len(vals)), vals, color=['#cc5544' if x>0 else '#318078' for x in vals])
    ax.axvline(0, color='black', lw=1)
    ax.set_yticks(np.arange(len(vals)),h.index)
    ax.invert_yaxis()
    ax.set_xlabel('Paired change in Hybrid MAE vs matcheed base (option price units)')
    ax.set_title('Robustness: error change when regime conditions shift')
    ax.grid(axis='x', alpha=.25)
    fig.tight_layout()
    fig.savefig(FIGURE_DIR/'hybrid_paired_stress_change.png', dpi=220, bbox_inches='tight')
    plt.show()
    plt.close(fig)